In [1]:
import pandas as pd
import json
import re

In [2]:
data_dirty = pd.read_csv('profi_masters_all_fields_new.csv')
data_dirty

,_Лет_на_сайте,_Ссылка,_Стаж,abbreviatedName,adaptiveReviews,albums,assembledInfoListing,audio,avatar,badges,...,scores,shortName,status,suitabilityHints,topServices,trustBadges,video,wantsBackofficeNoticies,wantsNewOrders,workplaces
0,—,https://profi.ru/profile/AndreevRS/,—,Роман А.,"{""review"": ""Роман просто волшебник!!!\nРабота ...","{""totalCount"": 2, ""edges"": [{""node"": {""id"": ""1...","[{""type"": ""UGC1"", ""content"": ""<p>Занимаюсь тол...","{""edges"": []}",//cdn.profi.ru/xfiles/pfiles/670d0727dae34ee68...,"[{""type"": ""verified"", ""text"": ""Паспорт провере...",...,"{""score"": null, ""pmetroWeight"": null}",Роман Андреев,active,[],"[{""name"": ""установка розеток и выключателей"", ...",[],NaN,"{""general"": false, ""suitableOrders"": false, ""c...","{""value"": ""YES"", ""status"": null, ""activeDate"":...",[]
1,—,https://profi.ru/profile/GromykoDS/,—,Дмитрий Г.,"{""review"": ""Все просто отлично.\nДмитрий приех...","{""totalCount"": 0, ""edges"": []}","[{""type"": ""UGC1"", ""content"": ""<p>Опыт&nbsp;— б...","{""edges"": []}",//cdn.profi.ru/xfiles/pfiles/c8e815c82ef94b36b...,"[{""type"": ""verified"", ""text"": ""Паспорт провере...",...,"{""score"": null, ""pmetroWeight"": null}",Дмитрий Громыко,active,[],"[{""name"": ""установка розеток и выключателей"", ...",[],NaN,"{""general"": false, ""suitableOrders"": false, ""c...","{""value"": ""YES"", ""status"": null, ""activeDate"":...",[]
2,—,https://profi.ru/profile/NedbailovAK/,—,Андрей Н.,"{""review"": ""Хочу выразить свою благодарность А...","{""totalCount"": 0, ""edges"": []}","[{""type"": ""UGC1"", ""content"": ""<p>Техник-электр...","{""edges"": []}",//cdn.profi.ru/xfiles/pfiles/8513ec63e3af4f578...,"[{""type"": ""verified"", ""text"": ""Паспорт провере...",...,"{""score"": null, ""pmetroWeight"": null}",Андрей Недбайлов,active,[],"[{""name"": ""установка розеток и выключателей"", ...",[],NaN,"{""general"": true, ""suitableOrders"": true, ""cal...","{""value"": ""YES"", ""status"": null, ""activeDate"":...",[]
3,—,https://profi.ru/profile/VarvinskiiVV/,—,Виталий В.,"{""review"": ""Огромное спасибо Виталию за качест...","{""totalCount"": 0, ""edges"": []}","[{""type"": ""UGC1"", ""content"": ""<p>Инженер-элект...","{""edges"": []}",//cdn.profi.ru/xfiles/pfiles/dae2dcf009874f7ab...,"[{""type"": ""verified"", ""text"": ""Паспорт провере...",...,"{""score"": null, ""pmetroWeight"": null}",Виталий Варвинский,active,[],"[{""name"": ""установка розеток и выключателей"", ...",[],NaN,"{""general"": true, ""suitableOrders"": false, ""ca...","{""value"": ""YES"", ""status"": null, ""activeDate"":...",[]
4,—,https://profi.ru/profile/LoktevVI/,—,Вячеслав Л.,"{""review"": ""Отличный мастер своего дела! Все у...","{""totalCount"": 0, ""edges"": []}","[{""type"": ""UGC1"", ""content"": ""<p>Квалифицирова...","{""edges"": []}",//cdn.profi.ru/xfiles/pfiles/ec173bf1fd4c48b7a...,"[{""type"": ""verified"", ""text"": ""Паспорт провере...",...,"{""score"": null, ""pmetroWeight"": null}",Вячеслав Локтев,active,[],"[{""name"": ""установка розеток и выключателей"", ...",[],NaN,"{""general"": true, ""suitableOrders"": false, ""ca...","{""value"": ""YES"", ""status"": null, ""activeDate"":...",[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,—,https://profi.ru/profile/ChernyiIN/,—,Иван Ч.,"{""review"": ""Был сложный случай - при капремонт...","{""totalCount"": 3, ""edges"": [{""node"": {""id"": ""1...","[{""type"": ""UGC1"", ""content"": ""<p>Среднеспециал...","{""edges"": []}",//cdn.profi.ru/xfiles/pfiles/74e90f83f0fd49c0a...,"[{""type"": ""verified"", ""text"": ""Паспорт провере...",...,"{""score"": null, ""pmetroWeight"": null}",Иван Черный,active,[],"[{""name"": ""установка розеток и выключателей"", ...",[],NaN,"{""general"": true, ""suitableOrders"": false, ""ca...","{""value"": ""YES"", ""status"": null, ""activeDate"":...",[]
156,—,https://profi.ru/profile/LuttsevSA2/,—,Сергей Л.,"{""review"": ""Сергей – спаситель от электро-ко

In [23]:
rows = []

for i in range(data_dirty.shape[0]):
    def get_json(column_name, index):
        val = data_dirty[column_name][index]
        if isinstance(val, str) and val.strip():
            try:
                return json.loads(val)
            except:
                return None
        return None

    try:
        # Проверка паспорта
        badges = get_json('badges', i)
        docs_verified = (badges[0]["type"] == 'verified') if (badges and len(badges) > 0) else False

        # Пол
        gender = data_dirty['gender'][i] if pd.notna(data_dirty['gender'][i]) else None

        # Гарантия
        guarantee_data = get_json('guaranteeBlock', i)
        master_garantee = None
        if guarantee_data and 'content' in guarantee_data and len(guarantee_data['content']) > 1:
            master_garantee = guarantee_data['content'][1].get('template')
        master_garantee = 'Нет гарантии' if not master_garantee else master_garantee
        if master_garantee.split()[0] == 'Советуем':
            master_garantee = 'По договоренности'
        if master_garantee == '«Сделаю скидку»':
            master_garantee = 'По договоренности'

        # Рейтинг и отзывы
        mean_rating = data_dirty['newRank'][i] if pd.notna(data_dirty['newRank'][i]) else 0
        reviews_count = data_dirty['reviewsCount'][i] if pd.notna(data_dirty['reviewsCount'][i]) else 0

        # Образование и опыт
        data_master = get_json('assembledInfoListing', i)
        experience = None
        last_education = None

        if data_master:
            temp_data = {"education": [], "experience": []}

            # Сортируем: блоки INFO в начало
            data_master_sorted = sorted(data_master, key=lambda x: x.get('type') != 'INFO')

            for item in data_master_sorted:
                content = item.get('content')
                if not content or not isinstance(content, str):
                    continue

                content = content.replace('&nbsp;', ' ')
                # Разделяем строки по тегам
                clean_text = re.sub(r'<(br|/p|/div|li)>', '\n', content)
                clean_text = re.sub(r'<[^>]+>', '', clean_text)

                lines = [line.strip() for line in clean_text.split('\n') if line.strip()]

                current_category = None
                for line in lines:
                    line = re.sub(r'^[•\-\*]\s*', '', line).strip()
                    line_lower = line.lower()

                    # Проверка на ОБРАЗОВАНИЕ (добавили синонимы)
                    edu_keywords = ["образование", "окончил", "диплом", "институт", "университет"]
                    if any(kw in line_lower for kw in edu_keywords):
                        current_category = "education"
                        # Если это не просто заголовок, а предложение - берем целиком
                        if ":" in line[:20]:
                            content_after = re.sub(r'^.*?образование[:\s-]*', '', line, flags=re.IGNORECASE).strip()
                            if content_after: temp_data["education"].append(content_after)
                        else:
                            temp_data["education"].append(line)
                        continue

                    # Проверка на ОПЫТ (добавили "стаж", "занимаюсь")
                    exp_keywords = ["опыт", "стаж", "работаю в сфере"] #"занимаюсь",
                    if any(kw in line_lower for kw in exp_keywords):
                        if item.get('type') == 'UGC1' and len(line) > 500: # Увеличили порог для длинных текстов
                            continue

                        current_category = "experience"
                        # Если в строке есть двоеточие- чистим начало
                        if ":" in line[:20] and "опыт" in line_lower[:20]:
                            content_after = re.sub(r'^.*?опыт\s*(работы)?[:\s-]*', '', line, flags=re.IGNORECASE).strip()
                            if content_after: temp_data["experience"].append(content_after)
                        else:
                            # Если это предложение типа "Мастер со стажем" - берем всё
                            temp_data["experience"].append(line)
                        continue

                    # Сбор данных внутри уже открытой категории
                    if current_category:
                        if line_lower.strip(': ') in ['опыт', 'опыт работы', 'образование', 'стаж']:
                            continue

                        # Дополнительный фильтр для опыта
                        if current_category == "experience" and len(line) > 300:
                            continue

                        temp_data[current_category].append(line)

            # Выборка финальных значений
            if temp_data['experience']:
                # Берем первую найденную значимую строку
                experience = temp_data['experience'][0]

            if temp_data['education']:
                last_education = temp_data['education'][0]
        # Отметка "очень хвалят"
        trust_badges = get_json('trustBadges', i)
        mark_hvalyat = (trust_badges[0]['type'] == 'great') if (trust_badges and len(trust_badges) > 0) else False

        # Проверка квалификации
        cert_data = get_json('certifiedServices', i)
        work_cert = (cert_data.get('title') == 'Квалификация подтверждена') if cert_data else False

        # 8. Цена
        price_list = get_json('priceListPreview', i)
        socket_price = None
        if price_list and 'prices' in price_list:
            for item in price_list['prices']:
                if item.get('price', {}).get('name') == 'установка розеток и выключателей':
                    socket_price = item['price'].get('from')
                    break

        # Сборка данных строки
        row = {
            "docs_verified": docs_verified,
            "gender": gender,
            "guarantee": master_garantee,
            "rating": mean_rating,
            "reviews_count": reviews_count,
            "experience": experience,
            "education": last_education,
            "is_highly_praised": mark_hvalyat,
            "is_certified": work_cert,
            "price_socket": socket_price
        }
        rows.append(row)

    except Exception as e:
        print(f"Критическая ошибка на строке {i}: {e}")
        rows.append({})

In [24]:
df_clean_2 = pd.DataFrame(rows)
df_clean_2

,docs_verified,gender,guarantee,rating,reviews_count,experience,education,is_highly_praised,is_certified,price_socket
0,True,MALE,«1 год»,"4,96",476,Частный опыт работы — 10 лет.,ФГОУ среднего профессионального образования «Ч...,False,True,400.0
1,True,MALE,«От 6 месяцев до 10 лет»,"4,98",482,None,"Кемеровский технический колледж, специальность...",False,True,350.0
2,True,MALE,«1 год»,"4,96",346,Опыт работы – с 2007 года.,"Строительно-монтажный колледж (г. Бельцы), рем...",False,True,300.0
3,True,MALE,«1 год»,"4,93",383,Опыт работы – с 2011 года.,Приазовский государственный технический универ...,False,True,300.0
4,True,MALE,«1 год»,"4,99",1786,Опыт профессиональной деятельности — 20 лет.,"БПГК, техник-электрик, 2000–2003 гг.",False,True,550.0
...,...,...,...,...,...,...,...,...,...,...
155,True,MALE,Нет гарантии,"4,98",188,"АО «МТЗ Рубин», 1998–2009 гг.","Сорокский техникум электрификации, 1988–1992 гг.",False,False,NaN
156,True,MALE,«В зависимости от услуги ( от 2х недель до год...,"5,0",296,Мастер по ремонту — 10 лет.,"Российский новый университет, техник-электрик,...",True,False,1000.0
157,True,MALE,По договоренности,"4,97",285,None,None,False,False,350.0
158,False,MALE,По договоренности,"4,96",163,None,"ГГТУ им. П.О. Сухого, электроснабжение, 2012–2...",True,False,250.0
